In [1]:
!pip install rouge-score nltk bert-score evaluate unsloth

# 1. Import Libraries

Top-3 Diagnostic Accuracy



Hallucination Rate



Medical Safety Score



Critical Symptom Recall



Faithfulness

Optional



BERTScore



Human Evaluation



Calibration score



In [2]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset

from openai import OpenAI

import json

from tqdm.auto import tqdm

import time

import re

import numpy as np
import pandas as pd

import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as edit_bert_score

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
nltk.download('punkt', quiet=True)

True

# 2. Load Test Dataset

In [ ]:
dataset_icliniq = load_dataset("lavita/ChatDoctor-iCliniq", split="train")

In [ ]:
split_dataset = dataset_icliniq.train_test_split(test_size=0.1, seed=3407)
raw_val_dataset = split_dataset["test"]

In [ ]:
def clean_text(text):
    if not text: return ""
    return re.sub(r'\s+', ' ', text).strip()

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert, professional, and deeply empathetic AI Medical Assistant. "
    "Analyze the patient's symptoms carefully, provide potential clinical insights, and suggest immediate precautions. "
    "Strictly Disclaimer: This is for informational purposes only, not a substitute for professional medical advice."
)

eval_samples = []
num_samples_to_extract = min(100, len(raw_val_dataset))

for i in range(num_samples_to_extract):
    item = raw_val_dataset[i]

    patient_query = clean_text(item["input"])

    doctor_response = item.get("answer_chatdoctor")
    if not doctor_response:
        doctor_response = item.get("answer_chatgpt") or item.get("answer_gpt")

    doctor_response = clean_text(doctor_response)

    if not patient_query or not doctor_response:
        continue

    eval_samples.append({
        "instruction": SYSTEM_PROMPT,
        "query": patient_query,
        "ground_truth": doctor_response
    })

with open("medical_eval_in_distribution.json", "w", encoding="utf-8") as f:
    json.dump(eval_samples, f, ensure_ascii=False, indent=4)

# 3. Load Model

In [ ]:
max_seq_length = 2048
dtype = None
load_in_4bit = True

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "ntq05/medical-llama3-lora",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
FastLanguageModel.for_inference(model)

==((====))==  Unsloth 2026.5.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-Instruct-bnb-4bit as a legacy tokenizer.


adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

Unsloth 2026.5.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [ ]:
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3.1-8b-instruct-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

FastLanguageModel.for_inference(base_model)

==((====))==  Unsloth 2026.5.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3.1-8b-instruct-bnb-4bit as a legacy tokenizer.


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm):

# 4. Generate Response

In [ ]:
def generate_response_base(query_text, max_new_tokens=256):

    messages = [
        {
            "role": "system",
            "content": "You are an expert, professional, and deeply empathetic AI Medical Assistant. Analyze the patient's symptoms carefully, provide potential clinical insights, and suggest immediate precautions. Strictly Disclaimer: This is for informational purposes only, not a substitute for professional medical advice."
        },
        {
            "role": "user",
            "content": query_text
        }
    ]

    inputs = base_tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = "pt",
    ).to("cuda")

    outputs = base_model.generate(
        input_ids = inputs,
        max_new_tokens = max_new_tokens,
        use_cache = True,
        temperature = 0.2,
        top_p = 0.9
    )

    input_length = inputs.shape[1]
    generated_tokens = outputs[0][input_length:]

    ai_response = base_tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    return ai_response

In [ ]:
def generate_response(query_text, max_new_tokens=256):

    messages = [
        {
            "role": "system",
            "content": "You are an expert, professional, and deeply empathetic AI Medical Assistant. Analyze the patient's symptoms carefully, provide potential clinical insights, and suggest immediate precautions. Strictly Disclaimer: This is for informational purposes only, not a substitute for professional medical advice."
        },
        {
            "role": "user",
            "content": query_text
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = "pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids = inputs,
        max_new_tokens = max_new_tokens,
        use_cache = True,
        temperature = 0.2,
        top_p = 0.9
    )

    input_length = inputs.shape[1]
    generated_tokens = outputs[0][input_length:]

    ai_response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    return ai_response

In [ ]:
try:
    with open("medical_eval_in_distribution.json", "r", encoding="utf-8") as f:
        dataset_test = json.load(f)
except FileNotFoundError:
    print("Error: Cannot find 'medical_eval_in_distribution.json'")
    dataset_test = []

evaluation_data = []

print("\n--- Generating Responses from Base and Fine-tuned LoRA Models ---")

num_samples = min(100, len(dataset_test))

for i in tqdm(range(num_samples)):
    item = dataset_test[i]

    system_prompt = item['instruction']
    query = item['query']
    gt = item['ground_truth']

    lora_res = generate_response(query)
    base_res = generate_response_base(query)

    evaluation_data.append({
        "index": i,
        "query": query,
        "ground_truth": gt,
        "lora_response": lora_res,
        "base_response": base_res
    })

with open("medical_eval_dataset.json", "w", encoding="utf-8") as f:
    json.dump(evaluation_data, f, ensure_ascii=False, indent=4)


--- Generating Responses from Base and Fine-tuned LoRA Models ---


  0%|          | 0/100 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn

# 4. Evaluation

In [5]:
try:
    with open("medical_eval_dataset.json", "r", encoding="utf-8") as f:
        evaluation_data = json.load(f)
except FileNotFoundError:
    print("Error: Cannot find 'medical_eval_dataset.json'")
    evaluation_data = []

In [6]:
queries = [item['query'] for item in evaluation_data]
ground_truths = [item['ground_truth'] for item in evaluation_data]
lora_responses = [item['lora_response'] for item in evaluation_data]
base_responses = [item['base_response'] for item in evaluation_data]

In [7]:
r_scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
smoothie = SmoothingFunction().method1

## 4.1 Lexical Evaluation

In [8]:
def calculate_lexical_metrics(refs, cands):
    r1_list, r2_list, rl_list, bleu_list = [], [], [], []

    for ref, cand in zip(refs, cands):
        r_scores = r_scorer.score(ref, cand)
        r1_list.append(r_scores['rouge1'].fmeasure)
        r2_list.append(r_scores['rouge2'].fmeasure)
        rl_list.append(r_scores['rougeL'].fmeasure)

        ref_tokens = ref.split()
        cand_tokens = cand.split()
        bleu = sentence_bleu([ref_tokens], cand_tokens, smoothing_function=smoothie)
        bleu_list.append(bleu)

    return {
        "ROUGE-1": np.mean(r1_list),
        "ROUGE-2": np.mean(r2_list),
        "ROUGE-L": np.mean(rl_list),
        "BLEU": np.mean(bleu_list)
    }

In [9]:
lora_lexical = calculate_lexical_metrics(ground_truths, lora_responses)
base_lexical = calculate_lexical_metrics(ground_truths, base_responses)

## 4.2 Semantic Evaluation

In [11]:
with torch.no_grad():
    P_lora, R_lora, F1_lora = edit_bert_score(
        lora_responses, ground_truths, lang="en", verbose=False,
        device="cuda" if torch.cuda.is_available() else "cpu"
    )
    P_base, R_base, F1_base = edit_bert_score(
        base_responses, ground_truths, lang="en", verbose=False,
        device="cuda" if torch.cuda.is_available() else "cpu"
    )

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
lora_bert_f1 = F1_lora.mean().item()
base_bert_f1 = F1_base.mean().item()

In [15]:
print("\n" + "="*20 + " PAIRED BENCHMARK REPORT (PURE METRICS) " + "="*20)

report_data = {
    "Metric Evaluation": ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU-Score", "BERTScore (F1-Semantic)"],
    "Base Model": [
        f"{base_lexical['ROUGE-1']:.4f}",
        f"{base_lexical['ROUGE-2']:.4f}",
        f"{base_lexical['ROUGE-L']:.4f}",
        f"{base_lexical['BLEU']:.4f}",
        f"{base_bert_f1:.4f}"
    ],
    "Fine-tuned LoRA": [
        f"{lora_lexical['ROUGE-1']:.4f}",
        f"{lora_lexical['ROUGE-2']:.4f}",
        f"{lora_lexical['ROUGE-L']:.4f}",
        f"{lora_lexical['BLEU']:.4f}",
        f"{lora_bert_f1:.4f}"
    ]
}

df_report = pd.DataFrame(report_data)
df_report


==================== PAIRED BENCHMARK REPORT (PURE METRICS) ====================


,Metric Evaluation,Base Model,Fine-tuned LoRA
0,ROUGE-1,0.2240,0.3410
1,ROUGE-2,0.0298,0.0926
2,ROUGE-L,0.1183,0.2041
3,BLEU-Score,0.0060,0.0431
4,BERTScore (F1-Semantic),0.8360,0.8613
